In [1]:
import pygame
import random

pygame.init()

# ================= SCREEN =================
WIDTH, HEIGHT = 900, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Boy Daily Journey Game")
clock = pygame.time.Clock()

# ================= COLORS =================
GREEN = (70, 180, 90)
GRAY = (120, 120, 120)
WHITE = (255, 255, 255)
RED = (200, 50, 50)
BROWN = (139, 69, 19)
BLUE = (60, 120, 220)
YELLOW = (240, 220, 140)
BLACK = (0, 0, 0)

# ================= FONTS =================
font = pygame.font.SysFont(None, 26)
big_font = pygame.font.SysFont(None, 42)

# ================= ROAD =================
road = pygame.Rect(360, 0, 180, HEIGHT)

# ================= PLAYER =================
boy_x, boy_y = 440, 500
boy_speed = 3

# ================= LOCATIONS =================
school = pygame.Rect(370, 10, 160, 60)
home = pygame.Rect(370, HEIGHT - 70, 160, 60)

market = pygame.Rect(90, 260, 140, 80)
friend = pygame.Rect(670, 260, 160, 80)

# ================= CARS =================
cars = []
lane_positions = [380, 420, 460, 500]
for _ in range(3):
    cars.append(
        pygame.Rect(
            random.choice(lane_positions),
            random.randint(-500, -120),
            30,
            60
        )
    )

car_speed = 2

# ================= TREES =================
trees = []
for y in range(0, HEIGHT, 120):
    trees.append(pygame.Rect(260, y, 25, 60))
    trees.append(pygame.Rect(590, y, 25, 60))

# ================= GAME STATE =================
paused = False
pause_reason = ""
journey = "HOME_TO_SCHOOL"
resume_position = None
win_menu = False

# ================= FUNCTIONS =================
def text(msg, x, y, big=False):
    img = big_font.render(msg, True, WHITE) if big else font.render(msg, True, WHITE)
    screen.blit(img, (x, y))

def draw_boy(x, y):
    pygame.draw.circle(screen, YELLOW, (x + 10, y - 5), 8)
    pygame.draw.rect(screen, BLUE, (x + 4, y + 5, 12, 18))
    pygame.draw.line(screen, BLACK, (x + 6, y + 23), (x + 6, y + 35), 2)
    pygame.draw.line(screen, BLACK, (x + 14, y + 23), (x + 14, y + 35), 2)

# ================= GAME LOOP =================
running = True
while running:
    screen.fill(GREEN)
    boy_rect = pygame.Rect(boy_x, boy_y, 20, 35)

    # ---------- EVENTS ----------
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        if paused and event.type == pygame.KEYDOWN:

            # WIN MENU OPTIONS
            if win_menu:
                if event.key == pygame.K_c:
                    paused = False
                    win_menu = False

                    if journey == "HOME_TO_SCHOOL":
                        journey = "SCHOOL_TO_HOME"
                        boy_x, boy_y = 440, 40
                    else:
                        journey = "HOME_TO_SCHOOL"
                        boy_x, boy_y = 440, 500

                elif event.key == pygame.K_e:
                    running = False

            # MARKET / FRIEND RESUME
            else:
                paused = False
                pause_reason = ""
                if resume_position:
                    boy_x, boy_y = resume_position
                    resume_position = None

    keys = pygame.key.get_pressed()

    # ---------- PLAYER MOVEMENT ----------
    if not paused:
        if keys[pygame.K_UP]:
            boy_y -= boy_speed
        if keys[pygame.K_DOWN]:
            boy_y += boy_speed
        if keys[pygame.K_LEFT]:
            boy_x -= boy_speed
        if keys[pygame.K_RIGHT]:
            boy_x += boy_speed

    boy_rect.topleft = (boy_x, boy_y)

    # ---------- SAFE ZONE ----------
    safe_zone = boy_rect.colliderect(home) or boy_rect.colliderect(school)

    # ---------- DRAW ROAD ----------
    pygame.draw.rect(screen, GRAY, road)
    pygame.draw.line(screen, WHITE, (450, 0), (450, HEIGHT), 4)

    # ---------- DRAW LOCATIONS ----------
    pygame.draw.rect(screen, BLUE, school)
    text("SCHOOL", 405, 30)

    pygame.draw.rect(screen, BROWN, home)
    text("HOME", 420, HEIGHT - 50)

    pygame.draw.rect(screen, YELLOW, market)
    text("MARKET", 120, 290)

    pygame.draw.rect(screen, YELLOW, friend)
    text("FRIEND", 710, 290)

    # ---------- TREES ----------
    for t in trees:
        pygame.draw.rect(screen, BROWN, t)
        pygame.draw.circle(screen, GREEN, (t.x + 12, t.y), 18)

    # ---------- CAR MOVEMENT ----------
    if not paused:
        for car in cars:
            car.y += car_speed
            if car.y > HEIGHT - 120:
                car.y = random.randint(-400, -120)
                car.x = random.choice(lane_positions)

            if boy_rect.colliderect(car) and not safe_zone:
                print("ACCIDENT! GAME OVER")
                running = False

    for car in cars:
        pygame.draw.rect(screen, RED, car)

    # ---------- COLLISIONS ----------
    if not paused:
        if boy_rect.colliderect(school) and journey == "HOME_TO_SCHOOL":
            paused = True
            win_menu = True
            pause_reason = "YOU REACHED SCHOOL 🎉"

        elif boy_rect.colliderect(home) and journey == "SCHOOL_TO_HOME":
            paused = True
            win_menu = True
            pause_reason = "YOU REACHED HOME 🎉"

        elif boy_rect.colliderect(market):
            paused = True
            pause_reason = "AT MARKET"
            resume_position = (boy_x, boy_y)

        elif boy_rect.colliderect(friend):
            paused = True
            pause_reason = "AT FRIEND HOUSE"
            resume_position = (boy_x, boy_y)

    # ---------- DRAW BOY ----------
    draw_boy(boy_x, boy_y)

    # ---------- UI ----------
    text("Cars move always | Home & School are safe | Road is dangerous", 160, 5)

    if paused:
        pygame.draw.rect(screen, BLACK, (200, 210, 500, 170))
        text(pause_reason, 300, 250, big=True)

        if win_menu:
            text("Press C : Go Again", 330, 300)
            text("Press E : Exit Game", 330, 330)
        else:
            text("Press ANY KEY to continue", 320, 320)

    pygame.display.update()
    clock.tick(60)

pygame.quit()


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
ACCIDENT! GAME OVER
